# LLM + Classical Planner: Neuro-Symbolic Task Planning

In this notebook we build a **real neuro-symbolic system**:

> A small open-source LLM + a classical planner (Pyperplan via Unified Planning)  
> working together to plan **smart home chores** from natural language instructions.

Pipeline:

```text
User request in natural language
        ↓
LLM (open source): parse into structured JSON (goals, rooms, options)
        ↓
Classical planner (Pyperplan): compute a valid plan that satisfies goals
        ↓
Pretty-printed plan + explanation
```


## 0. Background: Why LLM + Planner?

Classical planning lets us define **states, actions, and goals** and then automatically compute
a sequence of actions that achieves the goal, if one exists. This has been studied for decades
and is well formalized in the planning community.

Large language models, on the other hand, are very good at understanding **high-level intents** 
in natural language, but not so good at guaranteeing that a sequence of actions is **logically valid**
under a formal model of the world.

Recent work in robotics and task planning combines these ideas:

- Use an LLM to understand what the user wants (and sometimes to score actions).  
- Use a symbolic planner or value function to decide **how** to achieve it reliably.

In this notebook we implement a **mini version** of that idea for a smart-home chores domain:

> “Clean the kitchen, but do the dishes before mopping and take out the trash at the end.”

The LLM will parse this into structured goals, and Pyperplan will compute a valid plan
that respects preconditions and effects in a symbolic model.


## 1. Setup – Install and Configure Tools

We will use:

- `unified-planning` with the **Pyperplan** engine as our classical planner.  
- `transformers` from Hugging Face for the open-source LLM.

> 🔧 If you already have these installed in your environment, you can skip the install cell.


In [ ]:
# If you're on Colab or a fresh environment, run this once.
# If your environment already has these, you can skip.
!pip install -q 'unified-planning[pyperplan]' transformers accelerate sentencepiece

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import List, Dict, Any

from unified_planning.shortcuts import (
    BoolType,
    And,
    Object,
    Problem,
    UserType,
    Fluent,
    InstantaneousAction,
    OneshotPlanner,
)

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

import torch

### 1.1 Choose LLM Backend (Hugging Face in this case)

We’ll configure a **small open-source chat model** from Hugging Face.
You can swap this out for a bigger model or an Ollama endpoint if you prefer.

For this demo we use a relatively lightweight chat model so that it can run on modest hardware.
You can change `HF_MODEL_NAME` if you have more resources.


In [ ]:
HF_MODEL_NAME = "ibm-granite/granite-4.0-micro"  # you can swap to another open-source chat model

# Pick device: use GPU if available, otherwise CPU
if torch.backends.mps.is_available():
    device = "mps" # On M1/M2/M3 Macs, PyTorch uses Apple’s Metal backend (MPS) to access the GPU.
elif torch.cuda.is_available():
    device = "cuda" # For NVIDIA GPUs
else:
    device = "cpu"
print("Using device:", device)

# Chat-style text generation pipeline for JSON extraction
llm_pipe = pipeline(
    "text-generation",
    model=HF_MODEL_NAME,
    tokenizer=HF_MODEL_NAME,
    device=device,
    # Small max_new_tokens so JSON stays short
    max_new_tokens=256,
)

## 2. Symbolic World Model – Smart Home Chores Domain

We’ll now define a tiny **smart home domain** with a symbolic planner.

Entities:

- One or more **rooms** (kitchen, living_room, hallway, …)
- A **robot** that can be in a single room at a time
- Boolean facts (fluents):
  - `robot_at(room)` – where the robot is
  - `dishes_clean` – whether the dishes are done
  - `floor_clean(room)` – whether a room’s floor is clean
  - `trash_out` – whether trash has been taken out

Actions (simplified):

- `move(robot, from, to)` – move between rooms if connected  
- `wash_dishes` – requires robot in kitchen, makes `dishes_clean` true  
- `mop_floor(room)` – requires robot in that room and dishes already clean  
- `take_out_trash` – requires robot in kitchen, and after dishes

Preconditions & effects are encoded **symbolically**, and the planner is responsible for finding a sequence
of actions that satisfies the goal.


In [ ]:
from unified_planning.shortcuts import (
    BoolType,
    And,
    Object,
    Problem,
    UserType,
    Fluent,
    InstantaneousAction,
    OneshotPlanner,
    # Not  # <- we won't use Not in preconditions now
)

def build_smart_home_problem(
    rooms: List[str],
    connections: List[tuple],
    initial_robot_room: str,
    need_clean_dishes: bool,
    need_mop_rooms: List[str],
    need_trash_out: bool,
) -> Problem:
    """Create a classical planning problem for smart home chores."""

    # --- Types & Fluents ---
    Room = UserType("Room")

    robot_at = Fluent("robot_at", BoolType(), r=Room)
    connected = Fluent("connected", BoolType(), r1=Room, r2=Room)
    dishes_clean = Fluent("dishes_clean", BoolType())
    floor_clean = Fluent("floor_clean", BoolType(), r=Room)
    trash_out = Fluent("trash_out", BoolType())

    problem = Problem("smart_home_chores")
    problem.add_fluent(robot_at)
    problem.add_fluent(connected)
    problem.add_fluent(dishes_clean)
    problem.add_fluent(floor_clean)
    problem.add_fluent(trash_out)

    # --- Objects (rooms) ---
    room_objs: Dict[str, Object] = {}
    for room_name in rooms:
        obj = Object(room_name, Room)
        room_objs[room_name] = obj
        problem.add_object(obj)

    # --- Initial state ---
    for room_name, obj in room_objs.items():
        problem.set_initial_value(robot_at(obj), room_name == initial_robot_room)
        problem.set_initial_value(floor_clean(obj), False)

    for a, b in connections:
        ra, rb = room_objs[a], room_objs[b]
        problem.set_initial_value(connected(ra, rb), True)
        problem.set_initial_value(connected(rb, ra), True)

    # initially dishes are dirty, trash not out
    problem.set_initial_value(dishes_clean(), False)
    problem.set_initial_value(trash_out(), False)

    # --- Actions ---

    # Move between connected rooms
    move = InstantaneousAction("move", frm=Room, to=Room)
    frm = move.parameter("frm")
    to = move.parameter("to")
    move.add_precondition(connected(frm, to))
    move.add_precondition(robot_at(frm))
    move.add_effect(robot_at(frm), False)
    move.add_effect(robot_at(to), True)
    problem.add_action(move)

    # Wash dishes – only in kitchen
    # (No negative precondition; if dishes are already clean, applying again is harmless)
    wash_dishes = InstantaneousAction("wash_dishes")
    wash_dishes.add_precondition(robot_at(room_objs["kitchen"]))
    wash_dishes.add_effect(dishes_clean(), True)
    problem.add_action(wash_dishes)

    # Mop floor in a given room – requires robot there and dishes_clean
    # (No Not(floor_clean(r)) precondition; mopping twice is harmless)
    mop = InstantaneousAction("mop_floor", r=Room)
    r = mop.parameter("r")
    mop.add_precondition(robot_at(r))
    mop.add_precondition(dishes_clean())
    mop.add_effect(floor_clean(r), True)
    problem.add_action(mop)

    # Take out trash – only in kitchen, after dishes
    # (No Not(trash_out()) precondition; taking trash out twice is harmless)
    take_out = InstantaneousAction("take_out_trash")
    take_out.add_precondition(robot_at(room_objs["kitchen"]))
    take_out.add_precondition(dishes_clean())
    take_out.add_effect(trash_out(), True)
    problem.add_action(take_out)

    # --- Goals ---
    goals = []

    if need_clean_dishes:
        goals.append(dishes_clean())

    for room_name in need_mop_rooms:
        goals.append(floor_clean(room_objs[room_name]))

    if need_trash_out:
        goals.append(trash_out())

    if not goals:
        raise ValueError("At least one goal must be requested.")

    # If only one goal, And(*goals) is fine; UP will treat it as a simple conjunction
    problem.add_goal(And(*goals))

    return problem


### 2.1 Test the Planner Standalone

Before involving the LLM, let’s just test that the **symbolic domain** and planner work.

We’ll define a tiny apartment with two rooms: `kitchen` and `living_room`, connected both ways.
The robot starts in the `living_room`. The goal: 

- clean dishes  
- mop the kitchen floor  
- take out the trash


In [ ]:
rooms = ["kitchen", "living_room"]
connections = [("kitchen", "living_room")]

problem = build_smart_home_problem(
    rooms=rooms,
    connections=connections,
    initial_robot_room="living_room",
    need_clean_dishes=True,
    need_mop_rooms=["kitchen"],
    need_trash_out=True,
)

print(problem)

In [ ]:
# Solve with Pyperplan via Unified Planning
with OneshotPlanner(name="pyperplan", problem_kind=problem.kind) as planner:
    result = planner.solve(problem)

print("Status:", result.status)
print("Plan:")
print(result.plan)

If everything is installed correctly, you should see a **valid sequence of actions**, for example:

- move from living_room to kitchen  
- wash_dishes  
- mop_floor(kitchen)  
- take_out_trash  

The exact formatting may vary, but it should respect the preconditions we encoded.


## 3. LLM Front-End – Parsing Natural Language into Planning Goals

Now we let the **LLM handle the messy part**:  
understanding user instructions like:

> “Please clean the kitchen: do the dishes, mop the floor, and take the trash out.”

We’ll ask the model to output a **strict JSON schema** describing:

- which rooms exist
- where the robot starts
- which tasks are requested
- which rooms should be mopped


### 3.1 JSON Schema

We’ll use a simple schema like:

```json
{
  "rooms": ["kitchen", "living_room"],
  "initial_robot_room": "living_room",
  "tasks": {
    "clean_dishes": true,
    "mop_rooms": ["kitchen"],
    "take_out_trash": true
  }
}
```

The LLM’s job is to **map free-form text** into exactly this structure.
The planner’s job is to **achieve whatever the JSON requests**, if possible.


In [ ]:
PARSER_SYSTEM_PROMPT = """You are a strict JSON translator for smart-home chores.

Given a user request describing tasks for a home cleaning robot, you must extract
a JSON object with the following structure:

{
  "rooms": [list of room names as lowercase strings],
  "initial_robot_room": "one of the rooms",
  "tasks": {
    "clean_dishes": true or false,
    "mop_rooms": [list of room names where the floor must be cleaned],
    "take_out_trash": true or false
  }
}

Rules:
- Use ONLY lowercase letters and underscores in room names.
- If the user does not explicitly mention the starting room, default to "living_room".
- If the user does not mention some rooms, you can assume at least ["kitchen", "living_room"].
- Ensure the JSON is syntactically valid.
- Do not add any extra keys.
- Do not include comments, thinking or explanations, only pure JSON.
"""

In [ ]:
def extract_tasks_from_text(user_request: str) -> Dict[str, Any]:
    """Call the open-source LLM to map user text -> structured JSON.

    This uses the HF text-generation pipeline. For real applications you might switch
    to a chat-completions style interface, but the idea is the same.
    """
    prompt = PARSER_SYSTEM_PROMPT + "\n\nUser request:\n" + user_request + "\n\nJSON:"

    # Generate text from the model
    outputs = llm_pipe(
        prompt,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        num_return_sequences=1,
    )

    raw_text = outputs[0]["generated_text"][len(prompt) :].strip()
    print("Raw LLM output:\n", raw_text)

    # Heuristic: take the substring starting at first '{' and ending at last '}'.
    start = raw_text.find("{")
    end = raw_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"LLM response did not contain valid JSON:\n{raw_text}")

    json_str = raw_text[start : end + 1]

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse JSON from LLM output: {e}\nRaw JSON string:\n{json_str}")

    return data

> 🔍 In a production system, you might add more robust JSON repair, a schema validator,
or even a second LLM call to correct invalid JSON. For this educational notebook,
we keep it simple and fail loudly if the JSON is malformed.


## 4. Wiring LLM and Planner Together

We now connect the two worlds:

```text
Natural language
   ↓
LLM → JSON
   ↓
build_smart_home_problem(JSON)
   ↓
Pyperplan finds a plan
   ↓
We pretty-print the symbolic action list
```

This is a **classic neuro-symbolic pattern**:

- LLM = neural front-end for understanding instructions  
- Planner = symbolic back-end for computing valid action sequences


In [ ]:
@dataclass
class ParsedTaskConfig:
    rooms: List[str]
    initial_robot_room: str
    clean_dishes: bool
    mop_rooms: List[str]
    take_out_trash: bool


def normalize_llm_output(data: Dict[str, Any]) -> ParsedTaskConfig:
    """Validate and normalize the raw JSON from the LLM into a dataclass."""
    rooms = data.get("rooms", [])
    tasks = data.get("tasks", {})
    initial_robot_room = data.get("initial_robot_room", "living_room")

    if not rooms:
        rooms = ["kitchen", "living_room"]

    rooms = [r.strip().lower().replace(" ", "_") for r in rooms]
    if initial_robot_room not in rooms:
        # if invalid, default
        initial_robot_room = "living_room" if "living_room" in rooms else rooms[0]

    clean_dishes = bool(tasks.get("clean_dishes", False))
    mop_rooms = tasks.get("mop_rooms", [])
    mop_rooms = [r.strip().lower().replace(" ", "_") for r in mop_rooms if r in rooms]

    take_out_trash = bool(tasks.get("take_out_trash", False))

    # Some minimal sanity checks
    if clean_dishes is False and not mop_rooms and take_out_trash is False:
        raise ValueError("No tasks requested in JSON.")

    # Make sure kitchen exists if dishes/trash are required
    if (clean_dishes or take_out_trash) and "kitchen" not in rooms:
        rooms.append("kitchen")

    return ParsedTaskConfig(
        rooms=rooms,
        initial_robot_room=initial_robot_room,
        clean_dishes=clean_dishes,
        mop_rooms=mop_rooms,
        take_out_trash=take_out_trash,
    )

In [ ]:
def plan_from_request(user_request: str):
    """End-to-end pipeline: text -> LLM JSON -> symbolic plan."""
    print("USER REQUEST:\n", user_request)
    print("\n--- LLM: extracting structured tasks ---\n")
    raw = extract_tasks_from_text(user_request)
    print("LLM JSON:")
    print(json.dumps(raw, indent=2))

    config = normalize_llm_output(raw)
    print("\nNormalized task config:")
    print(config)

    # For simplicity, assume all rooms form a line: living_room <-> kitchen <-> hallway ...
    # We'll connect neighboring rooms in the given list.
    connections = []
    for i in range(len(config.rooms) - 1):
        connections.append((config.rooms[i], config.rooms[i + 1]))

    problem = build_smart_home_problem(
        rooms=config.rooms,
        connections=connections,
        initial_robot_room=config.initial_robot_room,
        need_clean_dishes=config.clean_dishes,
        need_mop_rooms=config.mop_rooms,
        need_trash_out=config.take_out_trash,
    )

    print("\n--- Symbolic planning problem ---\n")
    print(problem)

    print("\n--- Pyperplan: computing plan ---\n")
    with OneshotPlanner(name="pyperplan", problem_kind=problem.kind) as planner:
        result = planner.solve(problem)

    print("Status:", result.status)
    if not result.plan:
        print("No plan found.")
        return None

    print("\nRaw plan:")
    print(result.plan)

    # Pretty-print each action step on its own line
    print("\nPretty plan steps:")
    for i, action_instance in enumerate(result.plan.actions):
        print(f"Step {i+1}:", action_instance)  # each is an action instance with parameters

    return result.plan

## 5. Experiments – Ask for Plans in Natural Language

Let’s try a few natural language instructions and see how the **LLM + planner** system responds.

You can tweak the wording and see how robust the parser is.


In [ ]:
example_requests = [
    """
    Clean up the kitchen: do the dishes, mop the kitchen floor, and take the trash out.
    Start from the living room.
    """.strip(),
    """
    Just make sure the dishes are clean and the trash is out. Don't worry about mopping. Assume the robot starts in the kitchen.
    """.strip(),
]

for req in example_requests:
    print("=" * 80)
    plan_from_request(req)
    print("\n\n")

> 🔁 Try your own instructions: change rooms, omit some tasks, or give partial requests.
The LLM will attempt to fill in a reasonable JSON configuration, and the planner will do the rest.


## 6. Visual Summary – Neuro-Symbolic Planning Pattern

```text
           Natural language goal
      "Clean the kitchen and take out trash"
                          |
                          v
      ┌──────────────────────────────────┐
      │  LLM (open source, via HF)       │
      │  - understand intent             │
      │  - output strict JSON schema     │
      └──────────────────────────────────┘
                          |
                          v
      ┌──────────────────────────────────┐
      │  Symbolic Planner (Pyperplan)    │
      │  - smart_home_chores domain      │
      │  - preconditions & effects       │
      │  - search for valid plan         │
      └──────────────────────────────────┘
                          |
                          v
                Plan + textual explanation
```

The *neuro* part handles **language and ambiguity**.  
The *symbolic* part handles **validity and guarantees**.

This is similar in spirit to recent LLM+planning architectures used in robotics and task planning,
but distilled into a small, reproducible example.


## 7. Where to Go Next

If you want to extend this notebook, you can:

- Add more actions and constraints (time windows, limited resources, etc.).  
- Introduce **multiple goals and preferences**, then let the planner choose the best plan.  
- Use a stronger open-source model (e.g., a larger LLaMA or Mistral variant) if you have GPU access.  
- Integrate with a **robot simulator** or home automation mock API to actually execute the plan.

From a research and learning perspective, this notebook is a compact example of a **modern
neuro-symbolic architecture**:

> LLM for high-level understanding, symbolic planner for low-level precision.

